In [1]:
!pip install bcchapi holidays -q

In [2]:
import bcchapi
import pandas as pd
from datetime import date, timedelta
import holidays
from google.colab import userdata  # Importación clave para tus Secrets

# 1- Configuración de Fechas
def obtener_ultimo_dia_habil():
    hoy = date.today()
    feriados = holidays.Chile(years=[hoy.year - 1, hoy.year])
    for anio in [hoy.year - 1, hoy.year]:
        feriados[date(anio, 12, 31)] = "Feriado Bancario"

    fecha = hoy - timedelta(days=1)
    while fecha.weekday() >= 5 or fecha in feriados:
        fecha -= timedelta(days=1)
    return fecha

fecha_consulta = obtener_ultimo_dia_habil() + timedelta(days=1)

# 2- Conexión Segura Directa
# Rescatamos tu secreto único y lo dividimos
credenciales_crudas = userdata.get('BCCH_AUTH')
usuario, clave = credenciales_crudas.split("@@")

# Conectamos directamente pasando las variables
siete = bcchapi.Siete(usuario, clave)


# 3- Diccionario de Series del BCCH
series = {
    "USD": "F073.TCO.PRE.Z.D",    # Dólar observado
    "EUR": "F072.CLP.EUR.N.O.D",  # Euro
    "UF":  "F073.UFF.PRE.Z.D",    # Unidad de Fomento
    "JPY": "F072.CLP.JPY.N.O.D",  # Yen Japonés
    "GBP": "F072.CLP.GBP.N.O.D"   # Libra Esterlina
}


# 4- Extracción y Creación de Variables Dinámicas
indicadores_hoy = {}
for nombre, codigo in series.items():
    try:
        df = siete.cuadro(
            series=[codigo],
            nombres=[nombre],
            desde=fecha_consulta.strftime("%Y-%m-%d"),
            hasta=fecha_consulta.strftime("%Y-%m-%d"),
            frecuencia="D",
            observado={nombre: "last"}
        )
        # Extraemos el valor numérico del DataFrame
        valor = float(df.iloc[0, 0])
        indicadores_hoy[nombre] = valor
    except Exception as e:
        print(f"Advertencia: No se pudo obtener {nombre}. Detalle: {e}")
        indicadores_hoy[nombre] = None

# Variables independientes para cualquier uso posterior
USD_val = indicadores_hoy.get("USD")
EUR_val = indicadores_hoy.get("EUR")
UF_val  = indicadores_hoy.get("UF")
JPY_val = indicadores_hoy.get("JPY")
GBP_val = indicadores_hoy.get("GBP")


# 5- Panel Final para Visualización
df_panel = pd.DataFrame(list(indicadores_hoy.items()), columns=['Indicador', 'Valor'])
df_panel['Fecha'] = fecha_consulta.strftime('%d-%m-%Y')

# Formatear el panel para mejor visualización
print("\n" + "="*35)
print(f" PANEL DE INDICADORES AL {fecha_consulta.strftime('%d-%m-%Y')} ")
print("="*35)
print(df_panel.to_string(index=False))
print("="*35)

# Exportar el panel a un archivo Excel
nombre_archivo = f"Indicadores_BCCH_{fecha_consulta.strftime('%Y%m%d')}.xlsx"
df_panel.to_excel(nombre_archivo, index=False)
print(f"\n¡Archivo '{nombre_archivo}' generado con éxito!")


 PANEL DE INDICADORES AL 05-03-2026 
Indicador    Valor      Fecha
      USD   895.14 05-03-2026
      EUR  1041.22 05-03-2026
       UF 39819.01 05-03-2026
      JPY     5.70 05-03-2026
      GBP  1195.75 05-03-2026

¡Archivo 'Indicadores_BCCH_20260305.xlsx' generado con éxito!
